In [6]:
import argparse
import glob
import os
from dataclasses import dataclass
from typing import Optional, Tuple, Union

import numpy as np
import pandas as pd


# -------------------------------
# Physical constants and units
# -------------------------------
EV_TO_GEV = 1.0e-9
EV_TO_HZ = 2.417989242e14
M_TO_GEV_INV = 5.0677e15
GEV_PER_CM3_TO_GEV4 = 1.0 / 1.3e41
ALPHA_EM = 1.0 / 137.035999084
E_CHARGE = np.sqrt(4.0 * np.pi * ALPHA_EM)
SPEED_OF_LIGHT_M_PER_S = 299_792_458.0
TE011_BESSEL_DERIVATIVE_ZERO = 3.8317059702075125
TM_BESSEL_ZEROS = {
    "TM010": 2.404825557695773,
    "TM020": 5.520078110286311,
    "TM030": 8.653727912911013,
}

RHO_DM_GEV_PER_CM3 = 0.45
RHO_DM = RHO_DM_GEV_PER_CM3 * GEV_PER_CM3_TO_GEV4
C_MDM_TE011 = 0.22
DEFAULT_PLOT_WINDOW_HALF_WIDTH = 0.01
CONSTANT_MASS_RTOL = 1.0e-10
MASS_MAP_SAME_FREQUENCY = "same-frequency"
MASS_MAP_FIXED_TE011 = "fixed-te011"
MASS_MAP_SAME_TUNING_RATIO = "same-tuning-ratio"
DEFAULT_MASS_MAP = MASS_MAP_SAME_TUNING_RATIO
SOURCE_MODE_TM010 = "TM010"
SOURCE_MODE_TM020 = "TM020"
SOURCE_MODE_SIDECAR_MIXED = "sidecar-mixed"
SIDECAR_TM020_MIN_FREQUENCY_HZ = 6.5e9
SOURCE_MODE_TM110 = "TM110"
MDM_MODE_TE201 = "TE201"
MDM_MODE_TE021 = "TE021"
RECTANGULAR_MODE_INDICES = {
    SOURCE_MODE_TM110: (1, 1, 0),
    MDM_MODE_TE201: (2, 0, 1),
    MDM_MODE_TE021: (0, 2, 1),}


def cylinder_length_from_volume_liters(volume_liters: float, radius_m: float) -> float:
    return volume_liters * 1.0e-3 / (np.pi * radius_m**2)


@dataclass(frozen=True)
class CavityConfig:
    radius_m: Optional[float]
    c_axion: Optional[Union[float, Tuple[float, float]]]
    label: str
    note: str = ""
    length_m: Optional[float] = None
    source_mode: str = SOURCE_MODE_TM010

    @property
    def supports_simple_recast(self) -> bool:
        return (
            self.radius_m is not None
            and self.length_m is not None
            and self.c_axion is not None
        )

    @property
    def radius_gev_inv(self) -> float:
        if self.radius_m is None:
            raise ValueError(f"{self.label} does not define a simple cavity radius")
        return self.radius_m * M_TO_GEV_INV

    @property
    def te011_frequency_hz(self) -> float:
        if self.radius_m is None or self.length_m is None:
            raise ValueError(f"{self.label} does not define a simple TE011 frequency")
        radial_wavenumber = TE011_BESSEL_DERIVATIVE_ZERO / self.radius_m
        longitudinal_wavenumber = np.pi / self.length_m
        angular_wavenumber = np.hypot(radial_wavenumber, longitudinal_wavenumber)
        return SPEED_OF_LIGHT_M_PER_S * angular_wavenumber / (2.0 * np.pi)

    @property
    def tm010_frequency_hz(self) -> float:
        return self.tm_frequency_hz(SOURCE_MODE_TM010)

    def tm_frequency_hz(self, mode: str) -> float:
        if self.radius_m is None:
            raise ValueError(f"{self.label} does not define a simple {mode} frequency")
        if mode not in TM_BESSEL_ZEROS:
            raise ValueError(f"{self.label} has unsupported source mode {mode}")
        radial_wavenumber = TM_BESSEL_ZEROS[mode] / self.radius_m
        return SPEED_OF_LIGHT_M_PER_S * radial_wavenumber / (2.0 * np.pi)

    @property
    def te011_to_tm010_frequency_ratio(self) -> float:
        return self.te011_frequency_hz / self.tm010_frequency_hz

    def source_mode_for_frequency(self, frequency_hz: pd.Series) -> pd.Series:
        if self.source_mode == SOURCE_MODE_SIDECAR_MIXED:
            return pd.Series(
                np.where(
                    frequency_hz >= SIDECAR_TM020_MIN_FREQUENCY_HZ,
                    SOURCE_MODE_TM020,
                    SOURCE_MODE_TM010,
                ),
                index=frequency_hz.index,
            )
        return pd.Series(self.source_mode, index=frequency_hz.index)

    def source_frequency_hz(self, m_a_eV: pd.Series) -> Union[float, pd.Series]:
        source_modes = self.source_mode_for_frequency(m_a_eV * EV_TO_HZ)
        unique_modes = set(source_modes.unique())
        if len(unique_modes) == 1:
            return self.tm_frequency_hz(next(iter(unique_modes)))

        frequencies = pd.Series(index=m_a_eV.index, dtype=float)
        for mode in unique_modes:
            frequencies.loc[source_modes == mode] = self.tm_frequency_hz(mode)
        return frequencies

    @property
    def c_axion_central(self) -> float:
        if self.c_axion is None:
            raise ValueError(f"{self.label} does not define a simple axion form factor")
        if isinstance(self.c_axion, tuple):
            c_min, c_max = self.c_axion
            return np.sqrt(c_min * c_max)
        return self.c_axion

@dataclass(frozen=True)
class CavityConfigRec:
    """Rectangular-cavity configuration for a solenoidal magnetic field."""

    a_m: Optional[float]
    b_m: Optional[float]
    d_m: Optional[float]
    c_axion: Optional[Union[float, Tuple[float, float]]]
    label: str
    note: str = ""
    source_mode: str = SOURCE_MODE_TM110
    mdm_mode: str = MDM_MODE_TE201
    a_range_m: Optional[Tuple[float, float]] = None

    @property
    def supports_simple_recast(self) -> bool:
        return (
            self.a_m is not None
            and self.b_m is not None
            and self.d_m is not None
            and self.c_axion is not None
            and self.source_mode in RECTANGULAR_MODE_INDICES
            and self.mdm_mode in RECTANGULAR_MODE_INDICES
        )

    def mode_frequency_hz(
        self, mode: str, a_m: Optional[Union[float, pd.Series]] = None
    ) -> Union[float, pd.Series]:
        if self.a_m is None or self.b_m is None or self.d_m is None:
            raise ValueError(f"{self.label} does not define a rectangular geometry")
        if mode not in RECTANGULAR_MODE_INDICES:
            raise ValueError(f"{self.label} has unsupported rectangular mode {mode}")

        m_index, n_index, p_index = RECTANGULAR_MODE_INDICES[mode]
        width_m = self.a_m if a_m is None else a_m
        inverse_wavelength = np.sqrt(
            (m_index / width_m) ** 2
            + (n_index / self.b_m) ** 2
            + (p_index / self.d_m) ** 2
        )
        return 0.5 * SPEED_OF_LIGHT_M_PER_S * inverse_wavelength

    @property
    def source_reference_frequency_hz(self) -> float:
        return float(self.mode_frequency_hz(self.source_mode))

    @property
    def mdm_reference_frequency_hz(self) -> float:
        return float(self.mode_frequency_hz(self.mdm_mode))

    @property
    def mdm_to_source_reference_frequency_ratio(self) -> float:
        return self.mdm_reference_frequency_hz / self.source_reference_frequency_hz

    def a_for_source_frequency_hz(self, frequency_hz: pd.Series) -> pd.Series:
        """Infer the moving-wall dimension from each published source frequency."""
        if self.b_m is None or self.d_m is None:
            raise ValueError(f"{self.label} does not define a rectangular geometry")

        m_index, n_index, p_index = RECTANGULAR_MODE_INDICES[self.source_mode]
        inverse_a_squared = (
            (2.0 * frequency_hz / SPEED_OF_LIGHT_M_PER_S) ** 2
            - (n_index / self.b_m) ** 2
            - (p_index / self.d_m) ** 2
        )
        if m_index == 0 or (inverse_a_squared <= 0.0).any():
            raise ValueError(
                f"Cannot infer dimension a for {self.label} from {self.source_mode}"
            )
        return m_index / np.sqrt(inverse_a_squared)

    def mdm_to_source_frequency_ratio(self, m_a_eV: pd.Series) -> pd.Series:
        source_frequency_hz = m_a_eV * EV_TO_HZ
        a_m = self.a_for_source_frequency_hz(source_frequency_hz)
        return self.mode_frequency_hz(self.mdm_mode, a_m) / source_frequency_hz

    def mdm_length_gev_inv(self, m_a_eV: pd.Series) -> pd.Series:
        if self.b_m is None:
            raise ValueError(f"{self.label} does not define rectangular dimension b")
        a_m = self.a_for_source_frequency_hz(m_a_eV * EV_TO_HZ)
        return np.sqrt(a_m * self.b_m) * M_TO_GEV_INV

    def c_mdm(self, m_a_eV: pd.Series) -> pd.Series:
        """Analytic overlap for the symmetric-gauge rectangular mDM source."""
        if self.b_m is None:
            raise ValueError(f"{self.label} does not define rectangular dimension b")
        a_m = self.a_for_source_frequency_hz(m_a_eV * EV_TO_HZ)
        if self.mdm_mode == MDM_MODE_TE201:
            return 4.0 * a_m / (np.pi**4 * self.b_m)
        if self.mdm_mode == MDM_MODE_TE021:
            return 4.0 * self.b_m / (np.pi**4 * a_m)
        raise ValueError(f"No rectangular mDM overlap defined for {self.mdm_mode}")

    @property
    def c_axion_central(self) -> float:
        if self.c_axion is None:
            raise ValueError(f"{self.label} does not define a simple axion form factor")
        if isinstance(self.c_axion, tuple):
            c_min, c_max = self.c_axion
            return np.sqrt(c_min * c_max)
        return self.c_axion


CavityConfigLike = Union[CavityConfig, CavityConfigRec]

CAVITY_CONFIGS = {
    "ADMX": CavityConfig(0.21, 0.69, "legacy ADMX/SQUID", length_m=1.014),
    "ADMX2018": CavityConfig(0.21, 0.40, "ADMX Run 1A", length_m=1.014),
    "ADMX2019_1": CavityConfig(0.21, 0.40, "ADMX Run 1B, lower band", length_m=1.014),
    "ADMX2019_2": CavityConfig(0.21, 0.40, "ADMX Run 1B, upper band", length_m=1.014),
    "ADMX2021": CavityConfig(0.21, 0.40, "ADMX Run 1C", length_m=1.014),
    "ADMX2024": CavityConfig(0.21, 0.40, "ADMX 2022 scan", length_m=1.014),
    "ADMX2025": CavityConfig(0.2095, 0.40, "ADMX 1.1--1.3 GHz", length_m=1.014),
    "ADMX_Sidecar": CavityConfig(0.032, (0.04, 0.61), "ADMX Sidecar A--C", length_m=cylinder_length_from_volume_liters(0.38, 0.032), source_mode=SOURCE_MODE_SIDECAR_MIXED, ),
    "ADMX_Sidecar_AC": CavityConfig(0.032, (0.04, 0.49), "ADMX Sidecar A,C", length_m=cylinder_length_from_volume_liters(0.38, 0.032), source_mode=SOURCE_MODE_SIDECAR_MIXED,),
    "HAYSTAC_PhaseI": CavityConfig(0.051, 0.50, "HAYSTAC Phase I", length_m=0.254),
    "HAYSTAC_PhaseII_ab": CavityConfig(0.051, 0.43, "HAYSTAC Phase IIa/IIb", length_m=0.254),
    "HAYSTAC_PhaseII_cd": CavityConfig(0.051, 0.43, "HAYSTAC Phase IIc/IId", length_m=0.254),
    "QUAX": CavityConfig(0.0261 / 2.0, 0.589, "QUAX-aγ SCC, 2019", length_m=cylinder_length_from_volume_liters(0.036, 0.0261 / 2.0), ),
    "QUAX2019": CavityConfig(0.0261 / 2.0, 0.589, "QUAX-aγ SCC, 2019", length_m=cylinder_length_from_volume_liters(0.036, 0.0261 / 2.0), ),
    "QUAX2": CavityConfig(0.011, 0.69, "QUAX-aγ JPA, 2021", length_m=cylinder_length_from_volume_liters(0.081, 0.011),),
    "QUAX2021": CavityConfig(0.011, 0.69, "QUAX-aγ JPA, 2021", length_m=cylinder_length_from_volume_liters(0.081, 0.011), ),
    "QUAX3": CavityConfig(None, None, "QUAX high-Q dielectric cavity, 2022", "Uses published VC030 = 0.034 L; mode-specific mDM overlap required.",),
    "QUAX4": CavityConfig(None, None, "QUAX dielectric cavity with TWPA, 2023", "Uses published VC030 = 0.034 L; mode-specific mDM overlap required."),
    "QUAX5": CavityConfig(0.0135, 0.69, "QUAX-LNF tunable haloscope, 2024", length_m=cylinder_length_from_volume_liters(0.139, 0.0135), ),
    "QUAX2024": CavityConfig(0.0135, 0.69, "QUAX-LNF tunable haloscope, 2024", length_m=cylinder_length_from_volume_liters(0.139, 0.0135), ),
    "CAPP-MAX" : CavityConfig(0.137, 0.60, "CAPP-MAX, 2024", length_m=0.640 ),
    "CAPP-1" : CavityConfig(0.067, 0.52, "CAPP-1" , length_m=0.246),
    "CAPP-6" : CavityConfig(0.137, 0.60, "CAPP-1" , length_m=0.640),
    "CAPP-9" : CavityConfig(0.039, 0.50, "CAPP-1" , length_m=0.300),
    "GrAHal" : CavityConfig(0.018, 0.69, "GrAHal" , length_m=0.150),
    "ORGAN" : CavityConfig(0.016, 0.40, "ORGAN" , length_m=0.080),
    "ORGAN-1a" : CavityConfig(0.016, 0.40, "ORGAN-1a" , length_m=0.080),
}

CAVITY_CONFIGS_REC = {
    "ORGAN-1b": CavityConfigRec(a_m=0.0058, b_m=0.0271, d_m=0.0773, c_axion=0.40, label="ORGAN Phase 1b",
        source_mode=SOURCE_MODE_TM110, mdm_mode=MDM_MODE_TE201, a_range_m=(0.0056, 0.0060),
        note=("Mode-aware TE201 projection from the published TM110 scan; "
            "the published channel did not directly measure TE201." ),),}

def m_phi_from_axion_mass(
    m_a_eV: pd.Series,
    config: CavityConfigLike,
    mass_map: str,
    te_tm_frequency_ratio: Optional[float] = None,
) -> pd.Series:
    if mass_map == MASS_MAP_SAME_FREQUENCY:
        return m_a_eV / 2.0

    if mass_map == MASS_MAP_SAME_TUNING_RATIO:
        if te_tm_frequency_ratio is not None:
            frequency_ratio = te_tm_frequency_ratio
        elif isinstance(config, CavityConfigRec):
            frequency_ratio = config.mdm_to_source_frequency_ratio(m_a_eV)
        else:
            frequency_ratio = config.te011_frequency_hz / config.source_frequency_hz(
                m_a_eV
            )
        return frequency_ratio * m_a_eV / 2.0

    if mass_map == MASS_MAP_FIXED_TE011:
        if isinstance(config, CavityConfigRec):
            return pd.Series(
                config.mdm_reference_frequency_hz / (2.0 * EV_TO_HZ),
                index=m_a_eV.index,
            )
        return pd.Series(
            config.te011_frequency_hz / (2.0 * EV_TO_HZ),
            index=m_a_eV.index,)

    raise ValueError(f"Unknown mass mapping mode: {mass_map}")

def mass_mapping_header(
    mass_map: str,
    config: CavityConfigLike,
    te_tm_frequency_ratio: Optional[float] = None,
) -> list[str]:
    if mass_map == MASS_MAP_SAME_FREQUENCY:
        return [
            f"# mass_map = {MASS_MAP_SAME_FREQUENCY}",
            "# m_phi = m_a / 2 point by point at the published resonant frequency",
        ]

    if mass_map == MASS_MAP_SAME_TUNING_RATIO:
       if isinstance(config, CavityConfigRec):
            ratio = (
                te_tm_frequency_ratio
                if te_tm_frequency_ratio is not None
                else config.mdm_to_source_reference_frequency_ratio
            )
            return [
                f"# mass_map = {MASS_MAP_SAME_TUNING_RATIO}",
                f"# m_phi = (R_{config.mdm_mode}_{config.source_mode} / 2) m_a point by point",
                "# the movable a wall is inferred from each published source frequency",
                f"# source_mode = {config.source_mode}",
                f"# mDM_mode = {config.mdm_mode}",
                f"# R_{config.mdm_mode}_{config.source_mode}_ref = {ratio:g}",
                f"# f_{config.source_mode}_ref_Hz = {config.source_reference_frequency_hz:g}",
                f"# f_{config.mdm_mode}_ref_Hz = {config.mdm_reference_frequency_hz:g}",
            ]
    if te_tm_frequency_ratio is not None:
        ratio_lines = [f"# R_TE011_source = {te_tm_frequency_ratio:g} (user override)"]
    elif config.source_mode == SOURCE_MODE_SIDECAR_MIXED:
        ratio_lines = [
            "# source_mode = TM010 below 6.5 GHz, TM020 above 6.5 GHz",
            f"# R_TE011_TM010 = {config.te011_frequency_hz / config.tm_frequency_hz(SOURCE_MODE_TM010):g}",
            f"# R_TE011_TM020 = {config.te011_frequency_hz / config.tm_frequency_hz(SOURCE_MODE_TM020):g}",
            ]
    else:
        ratio_lines = [
            f"# source_mode = {config.source_mode}",
            f"# R_TE011_{config.source_mode} = {config.te011_frequency_hz / config.tm_frequency_hz(config.source_mode):g}",
            ]
    return [
        f"# mass_map = {MASS_MAP_SAME_TUNING_RATIO}",
        "# m_phi = (R_TE011_source / 2) m_a point by point",
        "# assumes the TE011 scan follows the same tuning trajectory as the published source-mode scan",
        *ratio_lines,
        f"# f_TM010_ref_Hz = {config.tm010_frequency_hz:g}",
        f"# f_TE011_ref_Hz = {config.te011_frequency_hz:g}",
    ]

    if mass_map == MASS_MAP_FIXED_TE011:
        if isinstance(config, CavityConfigRec):
            return [
                f"# mass_map = {MASS_MAP_FIXED_TE011}",
                f"# fixed rectangular mDM mode = {config.mdm_mode}",
                f"# f_{config.mdm_mode}_Hz = {config.mdm_reference_frequency_hz:g}",
            ]
        return [
            f"# mass_map = {MASS_MAP_FIXED_TE011}",
            "# m_phi = (f_TE011 / (2 f_TM_npq)) m_a with fixed ideal TE011 geometry",
            f"# f_TE011_Hz = {config.te011_frequency_hz:g}",
        ]

    raise ValueError(f"Unknown mass mapping mode: {mass_map}")


def epsilon_from_axion_limit(
    m_a_eV: pd.Series,
    g_agamma_GeV_inv: pd.Series,
    config: CavityConfigLike,
    c_axion: float,
    mass_map: str,
    te_tm_frequency_ratio: Optional[float] = None,
) -> pd.Series:
    """Convert an axion-photon limit into a fractional millicharge limit."""
    m_phi_eV = m_phi_from_axion_mass(
        m_a_eV, config, mass_map, te_tm_frequency_ratio
    )
    m_phi_GeV = m_phi_eV * EV_TO_GEV
    m_a_GeV = m_a_eV * EV_TO_GEV
    if isinstance(config, CavityConfigRec):
        c_ratio = c_axion / config.c_mdm(m_a_eV)
        mdm_length_gev_inv = config.mdm_length_gev_inv(m_a_eV)
    else:
        c_ratio = c_axion / C_MDM_TE011
        mdm_length_gev_inv = config.radius_gev_inv

    inside = (g_agamma_GeV_inv**2 * 8.0 * m_phi_GeV**5 * c_ratio / (mdm_length_gev_inv**2 * RHO_DM * m_a_GeV) )
    return np.power(inside, 0.25) / E_CHARGE

def make_plot_ready_limit(
    data: pd.DataFrame,
    value_columns: list[str],
    plot_window_half_width: float,
) -> tuple[pd.DataFrame, bool]:
    """Collapse a zero-width TE011 recast into a finite display window."""
    if data.empty or plot_window_half_width <= 0.0:
        return data.sort_values("m_phi_eV"), False

    clean = data[["m_phi_eV", *value_columns]].replace([np.inf, -np.inf], np.nan).dropna()
    if clean.empty:
        return clean, False

    masses = clean["m_phi_eV"].to_numpy()
    center = float(np.nanmedian(masses))
    spread = float(np.nanmax(masses) - np.nanmin(masses))
    scale = max(abs(center), 1.0e-300)
    if spread > CONSTANT_MASS_RTOL * scale:
        return clean.sort_values("m_phi_eV"), False

    lower = center * (1.0 - plot_window_half_width)
    upper = center * (1.0 + plot_window_half_width)
    collapsed = {"m_phi_eV": [lower, upper]}
    for column in value_columns:
        best_limit = float(clean[column].min())
        collapsed[column] = [best_limit, best_limit]
    return pd.DataFrame(collapsed), True


def read_axion_limit(path: str) -> pd.DataFrame:
    return pd.read_csv(
        path,
        comment="#",
        sep=r"\s+",
        header=None,
        names=["m_a_eV", "g_agamma_GeV_inv"],
    ).dropna()


def write_two_column_output(
    path: str,
    data: pd.DataFrame,
    config: CavityConfigLike,
    mass_map: str,
    plot_window_half_width: float,
    collapsed_mass_window: bool,
    te_tm_frequency_ratio: Optional[float],
) -> None:
    with open(path, "w", encoding="utf-8") as output:
        output.write("# Rescaled haloscope axion-photon limit to millicharged dark matter\n")
        output.write(f"# Source/run: {config.label}\n")
        for line in mass_mapping_header(mass_map, config, te_tm_frequency_ratio):
            output.write(f"{line}\n")
        if collapsed_mass_window:
            output.write("# constant-mass recast collapsed to the strongest limit for plotting\n")
            output.write(f"# plot_window_half_width_fraction = {plot_window_half_width:g}\n")
        if isinstance(config, CavityConfigRec):
            output.write(f"# a_reference_m = {config.a_m:g}\n")
            if config.a_range_m is not None:
                output.write(
                    f"# a_tuning_range_m = {config.a_range_m[0]:g}--{config.a_range_m[1]:g}\n"
                )
            output.write(f"# b_m = {config.b_m:g}\n")
            output.write(f"# d_m = {config.d_m:g}\n")
        else:
            output.write(f"# radius_m = {config.radius_m:g}\n")
            output.write(f"# length_m = {config.length_m:g}\n")
        output.write(f"# C_axion = {config.c_axion_central:g}\n")
        if isinstance(config, CavityConfigRec):
            c_mdm_ref = config.c_mdm(
                pd.Series([config.source_reference_frequency_hz / EV_TO_HZ])
            ).iloc[0]
            output.write(f"# C_mDM_{config.mdm_mode}_ref = {c_mdm_ref:g}\n")
            output.write(f"# note = {config.note}\n")
        else:
            output.write(f"# C_mDM_TE011 = {C_MDM_TE011:g}\n")
        output.write("# m_phi [eV]    epsilon = e_m/e\n")
        data.to_csv(output, sep="\t", index=False, header=False)


def write_range_output(
    path: str,
    data: pd.DataFrame,
    config: CavityConfigLike,
    mass_map: str,
    plot_window_half_width: float,
    collapsed_mass_window: bool,
    te_tm_frequency_ratio: Optional[float],
) -> None:
    with open(path, "w", encoding="utf-8") as output:
        output.write("# Rescaled ADMX Sidecar limit with reported C_axion range\n")
        output.write(f"# Source/run: {config.label}\n")
        for line in mass_mapping_header(mass_map, config, te_tm_frequency_ratio):
            output.write(f"{line}\n")
        if collapsed_mass_window:
            output.write("# constant-mass recast collapsed to the strongest limit for plotting\n")
            output.write(f"# plot_window_half_width_fraction = {plot_window_half_width:g}\n")
        if isinstance(config, CavityConfigRec):
            output.write(f"# a_reference_m = {config.a_m:g}\n")
            output.write(f"# b_m = {config.b_m:g}\n")
            output.write(f"# d_m = {config.d_m:g}\n")
        else:
            output.write(f"# radius_m = {config.radius_m:g}\n")
            output.write(f"# length_m = {config.length_m:g}\n")
        output.write(f"# C_axion range = {config.c_axion[0]:g}--{config.c_axion[1]:g}\n")
        if isinstance(config, CavityConfigRec):
            output.write(f"# C_mDM_mode = {config.mdm_mode}\n")
        else:
            output.write(f"# C_mDM_TE011 = {C_MDM_TE011:g}\n")
        output.write("# m_phi [eV]    epsilon(C_min)    epsilon(C_geom)    epsilon(C_max)\n")
        data.to_csv(output, sep="\t", index=False, header=False)



def process_file(
    file_path: str,
    output_dir: str,
    mass_map: str,
    plot_window_half_width: float,
    te_tm_frequency_ratio: Optional[float],
) -> bool:
    filename = os.path.basename(file_path)
    stem, ext = os.path.splitext(filename)
    config = CAVITY_CONFIGS.get(stem)

    if config is None:
        config = CAVITY_CONFIGS_REC.get(stem)

    if config is None:
        print(f"Skipping {filename}: no cavity configuration defined")
        return False

    if not config.supports_simple_recast:
        suffix = f": {config.note}" if config.note else ""
        print(f"Skipping {filename}: {config.label} needs a dedicated recast{suffix}")
        return False

    print(f"Processing {filename} with {config.label} parameters ...")
    df = read_axion_limit(file_path)
    df["m_phi_eV"] = m_phi_from_axion_mass(
        df["m_a_eV"], config, mass_map, te_tm_frequency_ratio
    )
    df["epsilon"] = epsilon_from_axion_limit(
        df["m_a_eV"],
        df["g_agamma_GeV_inv"],
        config,
        config.c_axion_central,
        mass_map,
        te_tm_frequency_ratio,
    )

    output_df, collapsed_mass_window = make_plot_ready_limit(
        df[["m_phi_eV", "epsilon"]], ["epsilon"], plot_window_half_width
    )
    output_path = os.path.join(output_dir, f"{stem}_rescaled{ext}")
    write_two_column_output(
        output_path,
        output_df,
        config,
        mass_map,
        plot_window_half_width,
        collapsed_mass_window,
        te_tm_frequency_ratio,
    )
    print(f"  -> Saved to {output_path}")

    if isinstance(config.c_axion, tuple):
        c_min, c_max = config.c_axion
        range_df = pd.DataFrame(
            {
                "m_phi_eV": df["m_phi_eV"],
                "epsilon_cmin": epsilon_from_axion_limit(
                    df["m_a_eV"],
                    df["g_agamma_GeV_inv"],
                    config,
                    c_min,
                    mass_map,
                    te_tm_frequency_ratio,
                ),
                "epsilon_cgeom": df["epsilon"],
                "epsilon_cmax": epsilon_from_axion_limit(
                    df["m_a_eV"],
                    df["g_agamma_GeV_inv"],
                    config,
                    c_max,
                    mass_map,
                    te_tm_frequency_ratio,
                ),
            }
        )
        range_output_df, range_collapsed_mass_window = make_plot_ready_limit(
            range_df,
            ["epsilon_cmin", "epsilon_cgeom", "epsilon_cmax"],
            plot_window_half_width,
        )
        range_path = os.path.join(output_dir, f"{stem}_rescaled_Crange{ext}")
        write_range_output(
            range_path,
            range_output_df,
            config,
            mass_map,
            plot_window_half_width,
            range_collapsed_mass_window,
            te_tm_frequency_ratio,
        )
        print(f"  -> Saved to {range_path}")

    return True


def main() -> None:
    parser = argparse.ArgumentParser(
        description="Rescale ADMX/HAYSTAC/QUAX AxionLimits data to millicharged dark matter."
    )
    parser.add_argument("--input-dir", default="AxionPhoton")
    parser.add_argument("--output-dir", default="Rescale")
    parser.add_argument(
        "--mass-map",
        choices=(
            MASS_MAP_SAME_FREQUENCY,
            MASS_MAP_SAME_TUNING_RATIO,
            MASS_MAP_FIXED_TE011,
        ),
        default=DEFAULT_MASS_MAP,
        help=(
            "Mass-coordinate mapping. 'same-frequency' keeps the usual broad "
            "haloscope recast m_phi=m_a/2. 'same-tuning-ratio' maps the "
            "published TM010 scan to a TE011 scan with a constant frequency "
            "ratio. 'fixed-te011' uses one ideal TE011 frequency from each "
            "configured radius and length."
        ),
    )
    parser.add_argument(
        "--te-tm-frequency-ratio",
        type=float,
        default=None,
        help=(
            "Optional constant R = f_TE011/f_TM010 used only with "
            "--mass-map same-tuning-ratio. If omitted, R is computed from the "
            "configured ideal cylinder geometry."
        ),
    )
    parser.add_argument(
        "--plot-window-half-width",
        type=float,
        default=DEFAULT_PLOT_WINDOW_HALF_WIDTH,
        help=(
            "Fractional half-width used only to display fixed-frequency recasts "
            "as finite mass windows. Use 0 to keep the raw constant-mass points."
        ),
    )
    args, unknown_args = parser.parse_known_args()
    if unknown_args:
        print(f"Ignoring unknown arguments: {' '.join(unknown_args)}")
    if args.te_tm_frequency_ratio is not None and args.te_tm_frequency_ratio <= 0.0:
        raise ValueError("--te-tm-frequency-ratio must be positive")
    if (
        args.te_tm_frequency_ratio is not None
        and args.mass_map != MASS_MAP_SAME_TUNING_RATIO
    ):
        print("Ignoring --te-tm-frequency-ratio because --mass-map is not same-tuning-ratio")

    os.makedirs(args.output_dir, exist_ok=True)
    paths = sorted(glob.glob(os.path.join(args.input_dir, "*.txt")))
    if not paths:
        print(f"No .txt files found in {args.input_dir}")
        return

    processed = sum(
        process_file(
            path,
            args.output_dir,
            args.mass_map,
            args.plot_window_half_width,
            args.te_tm_frequency_ratio,
        )
        for path in paths
    )
    print(f"Done. Processed {processed} supported haloscope file(s).")


if __name__ == "__main__":
    main()


Ignoring unknown arguments: -f C:\Users\usuario\AppData\Roaming\jupyter\runtime\kernel-0925a299-87b2-4948-a751-4f5a1fd5ff28.json
Skipping ABRACADABRA.txt: no cavity configuration defined
Skipping ABRACADABRA_run2.txt: no cavity configuration defined
Skipping ADBC.txt: no cavity configuration defined
Processing ADMX.txt with legacy ADMX/SQUID parameters ...
  -> Saved to Rescale\ADMX_rescaled.txt
Processing ADMX2018.txt with ADMX Run 1A parameters ...
  -> Saved to Rescale\ADMX2018_rescaled.txt
Processing ADMX2019_1.txt with ADMX Run 1B, lower band parameters ...
  -> Saved to Rescale\ADMX2019_1_rescaled.txt
Processing ADMX2019_2.txt with ADMX Run 1B, upper band parameters ...
  -> Saved to Rescale\ADMX2019_2_rescaled.txt
Processing ADMX2021.txt with ADMX Run 1C parameters ...
  -> Saved to Rescale\ADMX2021_rescaled.txt
Processing ADMX2024.txt with ADMX 2022 scan parameters ...
  -> Saved to Rescale\ADMX2024_rescaled.txt
Processing ADMX2025.txt with ADMX 1.1--1.3 GHz parameters ...
  ->